In [1]:
import sys
sys.path.insert(1, '../scripts/')
from preprocess import preprocess

First, format the project input files and generate the environment

In [2]:
# dp = '/data2/hratch/human_me/'
# preprocess.create_environment(build_path = dp + 'build/', input_path = dp + 'inputs/',
#                   outdir = dp + 'processed/', n_cores=20)

In [3]:
from preprocess import correct_inputs as ci
from utils.load_environmental_variables import *

full model

In [4]:
# prebuild = '/data2/hratch/human_me/prebuild/'
# ci.correct_model(model = prebuild + 'recon2_2.xml')
# non_machinery, revised_genes = ci.correct_psim(psim_df = input_data_path + 'psim_me.h5', fill_na = 'select', 
#                             non_machinery = {'HGNC:4556':['m', 'c'], 'HGNC:9251': ['l'], 
#                                             'HGNC:32043': ['e', 'n']})
# print(revised_genes)
# from expression import build_me_model
# me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False)
# me_model.pickle('/data2/hratch/human_me/full_12_29_20.pickle')

toy model

In [5]:
# other = '/data2/hratch/human_me/other/'
# ci.correct_model(model = other + 'toy_model.xml')
# revised_genes = ci.correct_psim(psim_df = input_data_path + 'psim_me.h5', fill_na = 'select', 
#                                non_machinery = None)
# print(revised_genes)
# from expression import build_me_model
# toy_me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False, 
#                                             unmodeled_protein_frac = None)
# toy_me_model.pickle(other + 'toy_me_12_29_20.pickle')
# sln, stat, _ = toy_me_model.solve_lp(mu_val = 1e-9)

In [4]:
from expression import build_me_model

minimal_proteome = True
compress_mrna = True
dummy_protein = True
deg_args = {'reversible_complex_formation': True, 
                       'couple': True, #False, 
                       'nonenzyme_degradation': False, # True 
                       'complex_degradation': True}


toy_me_model, builder = build_me_model.build_me(minimal_proteome = minimal_proteome, compress_mrna = compress_mrna, 
                                            dummy_protein = dummy_protein, deg_args = deg_args)


ERROR:cobra.io.sbml:No objective coefficients in model. Unclear what should be optimized


Generate ubiquitin reactions for proteasomal degradation
Generate ribosome


  0%|          | 0/591 [00:00<?, ?it/s]

Generate protein expression reactions for metabolic enzymes and non-machinery


100%|██████████| 591/591 [00:14<00:00, 41.36it/s]


...

 13%|█▎        | 160/1213 [00:00<00:00, 1599.83it/s]

Calculate enzyme k_effs


  0%|          | 0/489 [00:00<?, ?it/s]

A total of 2323 reactions were dropped when forming a minimal proteome
Add machinery to metabolic module reactions


AttributeError: 'Protein' object has no attribute 'alpha_p'

In [ ]:
counter = 7
lp_path = '/data2/hratch/human_me/other/test_lp/'
toy_me_model.pickle(lp_path + 'working_version_' + str(counter) + '.pickle')

# Check

In [11]:
# from expression import build_me_model
# toy_me_model, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False, 
#                                                 unmodeled_protein_frac = None,
#                                                 model_id = 'toy_me_model')
# jabba = True
# if jabba:
#     for r in toy_me_model.reactions:
#         if ('EX_' in r.id and r.compartments == {'b'} and r.bounds == (float('-inf'), float('inf'))):
#             r._lower_bound = -1000
#             r._upper_bound = 1000
            
# sln, stat, _ = toy_me_model.solve_lp(mu_val = 1e-9)

In [14]:
# toy_me_model.add_boundary(metabolite = toy_me_model.metabolites.get_by_id('h_c'), 
#                           type = 'demand')
# sln, stat, _ = toy_me_model.solve_lp(mu_val =  1e-9)

In [12]:
import pandas as pd
res_df = pd.DataFrame(data = {'reactions': [r.id for r in toy_me_model.reactions]})
res_df['fluxes'] = res_df.reactions.apply(lambda x: sln[toy_me_model.reactions.index(x)])
res_df.set_index(res_df.reactions, drop = True, inplace = True)
biom = res_df.loc[[i for i in res_df.index if 'biomass' in i],:]

biom.sort_values(by = 'fluxes', ascending = False)

,reactions,fluxes
reactions,,
biomass_dilution,biomass_dilution,1.000000e-09
DNA_biomass_formation,DNA_biomass_formation,1.000000e-09
carbohydrate_biomass_formation,carbohydrate_biomass_formation,1.000000e-09
lipid_biomass_formation,lipid_biomass_formation,1.000000e-09
protein_biomass_to_biomass,protein_biomass_to_biomass,5.384373e-10
mRNA_biomass_to_biomass,mRNA_biomass_to_biomass,2.130428e-10
lipid_biomass_to_biomass,lipid_biomass_to_biomass,9.700000e-11
carbohydrate_biomass_to_biomass,carbohydrate_biomass_to_biomass,7.100000e-11
unmodeled_protein_biomass_to_biomass,unmodeled_protein_biomass_to_biomass,6.648994e-11


In [10]:
counter = 3

S = toy_me_model.create_stoichiometric_matrix(mu_val = 1, inplace = False, array_type = 'pandas')
fn = '/data2/hratch/human_me/other/test_lp/S_matrix.h5'
if counter == 0:
    S.to_hdf(fn, key = str(counter), mode = 'w')
else:
    S.to_hdf(fn, key = str(counter), mode = 'a')

lp_path = '/data2/hratch/human_me/other/test_lp/'
toy_me_model.pickle(lp_path + 'working_version_' + str(counter) + '.pickle')

/home/hratch/Projects/human_me/me_env/lib/python3.6/site-packages/tables/path.py:155 NaturalNameWarning: object name is not a valid Python identifier: '3'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though


In [11]:
# tol = max([abs(i) for i in res['no_dummy']['infeasible_reactions'].values()])
# print('Tolerance: {}'.format(tol))
# fail = {k:v for k,v in res['dummy']['infeasible_reactions'].items() if abs(v) >= tol}

# fail_ids = set(pd.Series(list(fail.keys())).apply(lambda x: x.split('_')[0]))
# fail_metabs = [m for m in res['dummy']['model'].metabolites if m.id.split('_')[0] in fail_ids]